# Auto benchmark analysis

In [47]:
import math
import os
import json5
import prettytable
import numpy as np

# Directory Management
try:
    # Run in Terminal
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    # Run in ipykernel & interactive
    ROOT_DIR = os.getcwd()
    
AUTOBENCHMARK_DIR = os.path.join(ROOT_DIR, "data", "AutoBenchmarkOutput.json")
benchmark_result = json5.load(open(AUTOBENCHMARK_DIR, "r"))

class BenchmarkAnalysis:
    def __init__(self, benchmark_result : dict):
        self.benchmark_result = benchmark_result
    
    def print_table(self, variable : str):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        # create array
        results = np.zeros((planner_num, demo_num))
        for planner, demos in self.benchmark_result.items():
            for demo, benchmarks in demos.items():
                for benchmark, value in benchmarks.items():
                    if benchmark == variable:
                        results[list(self.benchmark_result.keys()).index(planner), list(demos.keys()).index(demo)] = value
        
        # create table
        results = results.tolist()
        table = prettytable.PrettyTable()
        table.field_names =["planner-"+variable]+ demo_names
        for i in range(planner_num):
            table.add_row([planner_names[i]] + [results[i][j] for j in range(demo_num)])
        # set table display precision
        table.float_format = ".3"
        print(table)
        
    def get_summary(self):
        planner_num = len(self.benchmark_result)
        planner_names = list(self.benchmark_result.keys())
        demo_names = list(self.benchmark_result[planner_names[0]].keys())
        demo_num = len(list(self.benchmark_result.values())[0])
        table = prettytable.PrettyTable()
        table.field_names = ["planner", "total num", "suc num", "ave.", "min.", "max.", "std.", "success rate", "ave traj len", "ave traj ctrl"]
        planner_names_correct = {"flt_cfg_planner_fast":"proposed(IKTS)",
                                 "rrt_cfg_planner":"RRT-Connect",
                                 "minco_cfg_planner":"MINCO+LBFGS",
                                 "stomp_cfg_planner":"STOMP",}
        for planner, demos in self.benchmark_result.items():
            tot_time = 0
            tot_time2 = 0
            tot_optnum = 0
            tot_successnum = 0
            tot_len = 0
            tot_ctrl = 0
            min_time = math.inf
            max_time = 0
            std_time = 0
            for demo, benchmarks in demos.items():
                tot_optnum += benchmarks["OptNum"]
                tot_successnum += benchmarks["OptNum"] * benchmarks["SuccessRate"]
                tot_time += benchmarks["Totaltime"]
                # tot_time2 += benchmarks["AveTime2"] * benchmarks["OptNum"]
                tot_len += benchmarks["AveLen"] * benchmarks["OptNum"]
                tot_ctrl += benchmarks["AveCtrl"] * benchmarks["OptNum"]
                if benchmarks["MinTime"] < min_time and benchmarks["MinTime"] != 0:
                    min_time = benchmarks["MinTime"]
                if benchmarks["MaxTime"] > max_time:
                    max_time = benchmarks["MaxTime"]
            ave_time = tot_time / tot_optnum
            ave_len = tot_len / tot_optnum
            ave_ctrl = tot_ctrl / tot_optnum
            # std = sqrt(expectation of square - square of expectation)
            # std_time = math.sqrt(tot_time2 / tot_optnum - ave_time**2)
            ave_successrate = tot_successnum / tot_optnum
            table.add_row([planner_names_correct[planner], tot_optnum, tot_successnum, ave_time, min_time, max_time, std_time, ave_successrate, ave_len, ave_ctrl])
        table.float_format = ".3"
        return table
    
    def print_summary(self):
        print(self.get_summary())
        
    def save_summary_csv(self, filename : str):
        table = self.get_summary()
        with open(filename, "w") as f:
            string = table.get_csv_string()
            string = string.replace("\n", "")
            f.write(string)
                
                
        

In [51]:
analysis = BenchmarkAnalysis(benchmark_result)
# analysis.print_table("SuccessRate")
# analysis.print_table("Totaltime")
# analysis.print_table("AveTime")
# analysis.print_table("MaxTime")
# analysis.print_table("StdTime")
# analysis.print_table("AveLen")
# analysis.print_table("AveCtrl")
analysis.print_summary()
analysis.save_summary_csv(os.path.join(ROOT_DIR, "data", "benchmark_summary.csv"))

+----------------+-----------+---------+-------+-------+--------+------+--------------+--------------+---------------+
|    planner     | total num | suc num |  ave. |  min. |  max.  | std. | success rate | ave traj len | ave traj ctrl |
+----------------+-----------+---------+-------+-------+--------+------+--------------+--------------+---------------+
| proposed(IKTS) |    504    | 497.000 | 0.814 | 0.213 | 7.016  |  0   |    0.986     |    1.735     |     68.544    |
|  MINCO+LBFGS   |    614    | 497.000 | 1.926 | 0.454 | 9.030  |  0   |    0.809     |    3.047     |    120.942    |
|  RRT-Connect   |    497    | 497.000 | 1.684 | 0.269 | 20.335 |  0   |    1.000     |    2.488     |    1504.745   |
|     STOMP      |    334    | 254.000 | 5.285 | 0.191 | 21.234 |  0   |    0.760     |    1.172     |    273.378    |
+----------------+-----------+---------+-------+-------+--------+------+--------------+--------------+---------------+
